##### Generating RAG answers

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
        
documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
# Running RAG
from openai import OpenAI

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client
)

In [6]:
# For each question, RAGBase searches the FAQ, builds a prompt, and calls the LLM to get an answer
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

"Based on the provided context, to answer your question:\n\nIf I miss the deadline for submissions in a live cohort (not self-paced mode) and still aim to get a certificate, the following applies:\n\n- Missing any homework will not affect your chances of getting the certificate as long as you pass the Capstone project.\n- You can still submit late if you do so while the submission form is open, but you won't be able to submit late after the form has been closed.\n\nTherefore, as long as the Capstone project's requirements are met, missing a homework deadline in itself isn't enough to prevent you from getting your certificate."

In [7]:
assistant.total_cost()

0.0009547500000000001

In [8]:
# Get the original answer from the document ID
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_original = original_doc["answer"]

answer_original

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [9]:
# Save both answers in one record
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_original": answer_original,
    "document": doc_id
}

rag_result

{'question': 'If I miss the deadline for submissions, can I still get a certificate?',
 'answer_llm': "Based on the provided context, to answer your question:\n\nIf I miss the deadline for submissions in a live cohort (not self-paced mode) and still aim to get a certificate, the following applies:\n\n- Missing any homework will not affect your chances of getting the certificate as long as you pass the Capstone project.\n- You can still submit late if you do so while the submission form is open, but you won't be able to submit late after the form has been closed.\n\nTherefore, as long as the Capstone project's requirements are met, missing a homework deadline in itself isn't enough to prevent you from getting your certificate.",
 'answer_original': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

##### Processing all Questions

In [10]:
# Create a function that processes one ground truth record
def generate_rag_answer(rec):
    question = rec["question"]
    answer_llm = assistant.rag(question)
    
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]
    answer_original = original_doc["answer"]
    
    rag_result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_original": answer_original,
        "document": doc_id
    }
    
    return rag_result

In [11]:
# Test it
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'If I miss the deadline for submissions, can I still get a certificate?',
 'answer_llm': "Based on the provided context, to answer your original question:\n\nIf you miss the deadline for submissions, but you completed a Capstone project within the timeframe of the live cohort, you may still be eligible to receive a certificate. Homework is not mandatory, but completing a Capstone project is required to get the certificate.\n\nHowever, if you only missed the homework submission deadline and didn't complete the Capstone project or peer reviews, you won't be able to get a certificate.\n\nKeep in mind that the context doesn't explicitly address what happens exactly when you miss the deadline, so the answer might vary depending on individual course circumstances.",
 'answer_original': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
# reset usage
assistant.reset_usage()

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [14]:
# Run RAG for 1/5 of the ground truth records in parallel for speed
ground_truth_subset = ground_truth[:len(ground_truth)//5]

with ThreadPoolExecutor(max_workers=6) as executor:
    rag_results = map_progress(executor, ground_truth_subset, generate_rag_answer)

  0%|          | 0/140 [00:00<?, ?it/s]

In [15]:
rag_results[:10]

[{'question': 'If I miss the deadline for submissions, can I still get a certificate?',
  'answer_llm': 'To answer your question directly: \n\nYes, you CAN still get a certificate.\n\nThe text does not mention missing a homework submission, but getting an extension for one is mentioned in this section : A: "No. We don\'t give individual deadline extensions, and once the homework submission form is closed you can no longer submit it — there are no late submissions."\n\nHowever the exact question asked is If I MISS THE DEADLINE FOR SUBMISSIONS...',
  'answer_original': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'What is the format of the project submissions and what are the required materials?',
  'answer_llm': "I'll answer questions based on the provided context.\n\n1. To access the current workshop materials for the Open-Source Data Ingestion (DLT) workshop, you ca

In [18]:
# Save the results to a CSV file
df_answers = pd.DataFrame(rag_results)
df_answers.to_csv("data/rag_results.csv", index=False)

In [21]:
# Check the dataframe
df_answers.head()

,question,answer_llm,answer_original,document
0,"If I miss the deadline for submissions, can I ...","To answer your question directly: \n\nYes, you...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,What is the format of the project submissions ...,I'll answer questions based on the provided co...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,Can someone who has already joined the course ...,"Yes, someone who has already joined the course...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,How long after submitting my project will I re...,I don't know how long after submitting my proj...,"Yes, but if you want to receive a certificate,...",74eb249bbf
4,Will there be any penalties or late fees for m...,Based on the provided context:\n\nThere won't ...,"Yes, but if you want to receive a certificate,...",74eb249bbf


In [20]:
# calculate the total cost of the RAG calls
assistant.total_cost()

0.16737000000000005